In [ ]:
import numpy as np
import pandas as pd



In [ ]:
df=pd.read_csv('combined_dataset_1hour.csv',parse_dates=["datetime"],)
df.head(15)


In [ ]:
# ohlc_mean = df[
#     ["stock_open", "stock_high", "stock_low", "stock_close"]
# ].mean(axis=1)

In [ ]:
# previous_close = df.groupby("ticker")["stock_close"].shift(1)

# df["stock_OHLC_mean_ratio"] = ohlc_mean / previous_close - 1

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
bar_size = "1hour"
df=pd.read_csv(f'combined_dataset_{bar_size}.csv',parse_dates=["datetime"],)
feature_columns = [
    column
    for column in df.columns
    if column.startswith(("stock_", "smh_", "spy_"))
    and not column.endswith((
        "_open",
        # "_close",
        "_high",
        "_low",
        # "_volume",
        "_Direction",
        "_Return",
        "_LogReturn",
        "ock_Count15",    
        "mh_Count15",
        "_ShadowImbalance",   
    ))
]
# feature_columns.append("datetime")
# feature_columns.append("ticker")
# feature_columns.append("ticker_id")

In [ ]:
df.describe()

In [ ]:
# feature_data = df[feature_columns]  # Use correlation_columns instead of feature_columns

feature_report = pd.DataFrame({
    "nan_count": df.isna().sum(),
    "positive_inf": df.eq(np.inf).sum(),
    "negative_inf": df.eq(-np.inf).sum(),
    "min": df.replace(
        [np.inf, -np.inf], np.nan
    ).min(),
    "max": df.replace(
        [np.inf, -np.inf], np.nan
    ).max(),
    'empty_string_count': (df == '').sum(),
})
display(
    feature_report[
        (feature_report["nan_count"] > 0)
        | (feature_report["positive_inf"] > 0)
        | (feature_report["negative_inf"] > 0)
    ]
)


In [ ]:
shadow_mode = df['stock_ShadowImbalance'].mode()
fill_value = shadow_mode[0] if not shadow_mode.empty else 0

df['stock_ShadowImbalance'].fillna(fill_value, inplace=True)

In [ ]:
print("Remaining NaNs:", df['stock_ShadowImbalance'].isna().sum())

In [ ]:
eature_report = pd.DataFrame({
    "nan_count": df.isna().sum(),
    "positive_inf": df.eq(np.inf).sum(),
    "negative_inf": df.eq(-np.inf).sum(),
    "min": df.replace(
        [np.inf, -np.inf], np.nan
    ).min(),
    "max": df.replace(
        [np.inf, -np.inf], np.nan
    ).max(),
    'empty_string_count': (df == '').sum(),
})
display(
    feature_report[
        (feature_report["nan_count"] > 0)
        | (feature_report["positive_inf"] > 0)
        | (feature_report["negative_inf"] > 0)
    ]
)

In [ ]:
problem_rows = df[df["stock_ShadowImbalance"].isna()]

ticker_report = (
    problem_rows
    .groupby("ticker")
    .size()
    .rename("missing_count")
    .reset_index()
    .sort_values("missing_count", ascending=False)
)

display(ticker_report)

print(
    "Distinct stocks with missing stock_ShadowImbalance:",
    problem_rows["ticker"].nunique(),
    "out of",
    df["ticker"].nunique()
)

In [ ]:
missing_counts = (df.isna()) | (df == 'NaN') | (df == '') | (df == np.inf) | (df == -np.inf)

# Print the total count per column, sorted by the most missing
print(missing_counts.sum().sort_values(ascending=False))
most_frequent_value = df['stock_LogReturn'].mode()[0]
print(f"The most frequent value is: {most_frequent_value}")

In [ ]:
print(feature_data["stock_LogReturn"].value_counts(dropna=False))

In [ ]:
print(
    df["stock_GapPct_Missing"]
    .value_counts(dropna=False)
)

print(
    df["stock_RollingStd15_Missing"]
    .value_counts(dropna=False)
)
print(
    df["stock_RollingStd15_Missing"]
    .value_counts(dropna=False)
)

In [ ]:

# correlation_matrix = df[feature_columns].corr()
correlation_matrix = df[feature_columns].corr(numeric_only=True,method='pearson')
    
plt.figure(figsize=(20, 15))

sns.heatmap(
        correlation_matrix,
        annot=True,       # Display correlation numbers
        fmt=".2f",        # Show two decimal places
        cmap="coolwarm",  # Blue-white-red colors
        center=0,
        vmin=-1,
        vmax=1
    )

plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()